<a href="https://colab.research.google.com/github/fe2val/Education_UP/blob/main/temperature_study2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U google-genai

In [ ]:
from google import genai
from google.genai import types
import ipywidgets as widgets
import asyncio
import nest_asyncio
from IPython.display import display, Markdown
from google.colab import userdata

# Allow the already-running Colab event loop to be re-entered
nest_asyncio.apply()

# 1. Initialize the new Client using your secure key
try:
    api_key = userdata.get('GOOGLE_API_KEY').strip()
    client = genai.Client(api_key=api_key)
except userdata.SecretNotFoundError:
    print("⚠️ ERROR: You forgot to add your API key!")
    raise SystemExit

# 2. Build the interactive UI elements
prompt_input = widgets.Textarea(
    value='Suggest a creative name for a pet dragon and explain why.',
    description='Prompt:',
    layout=widgets.Layout(width='80%', height='80px')
)

temp_slider = widgets.FloatSlider(
    value=1.0, min=0.0, max=2.0, step=0.1,
    description='Temperature:', layout=widgets.Layout(width='50%')
)

num_gens_slider = widgets.IntSlider(
    value=3, min=1, max=5, step=1,
    description='Responses:', layout=widgets.Layout(width='50%')
)

generate_btn = widgets.Button(
    description='Generate in Parallel ⚡',
    button_style='success'
)

output_area = widgets.Output()

# 3. Define the ASYNC logic using the new SDK syntax
async def fetch_responses_async():
    with output_area:
        output_area.clear_output()
        print(f"⚡ Firing off {num_gens_slider.value} requests simultaneously. Please wait...\n")

        try:
            tasks = []
            for _ in range(num_gens_slider.value):
                task = client.aio.models.generate_content(
                    model='gemini-2.5-flash',
                    contents=prompt_input.value,
                    config=types.GenerateContentConfig(
                        temperature=temp_slider.value,
                    )
                )
                tasks.append(task)

            responses = await asyncio.gather(*tasks)

            for i, response in enumerate(responses):
                display(Markdown(f"### 🟢 Response {i + 1}"))
                display(Markdown(response.text))
                display(Markdown("--- \n"))

        except Exception as e:
            output_area.clear_output()
            print(f"Error: {e}")

# 4. Button callback — run the coroutine to completion on the patched loop
def on_button_click(b):
    asyncio.get_event_loop().run_until_complete(fetch_responses_async())

generate_btn.on_click(on_button_click)

# 5. Display the UI
display(prompt_input, temp_slider, num_gens_slider, generate_btn, output_area)